In [ ]:
# import kagglehub

# from dotenv import load_dotenv

# load_dotenv()

# # Download latest version
# path = kagglehub.competition_download("unipd-deep-learning-2026-challenge-1")

# print("Path to competition files:", path)



In [ ]:
%load_ext autoreload
%autoreload 2

import os
import copy
from pathlib import Path
from timeit import default_timer as timer
from typing import Callable

import hiddenlayer as hl
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import torch
import torch.nn.functional as F
from PIL import Image
from torch import nn
from torch.nn import Conv2d, MaxPool2d, Linear
from torch.utils.data import DataLoader, Dataset, random_split
from torchinfo import summary
from torchview import draw_graph
from torchvision import transforms
from torchvision.transforms import Lambda

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(4242)


## Dataset and Dataloader

In [ ]:
class CelebDataset(Dataset):
    def __init__(self, img_dir: Path, csv_file: Path | None = None, transform=None, test: bool = False):
        self.img_dir = img_dir
        self.transform = transform
        self.test = test

        if not test:
            assert csv_file is not None, "csv_file is required when test=False"
            self.data_frame = pl.read_csv(
                csv_file,
                schema={
                    "id": pl.String,
                    "No_Beard": pl.Int8,
                    "Young": pl.Int8,
                    "Mouth_Slightly_Open": pl.Int8,
                    "Smiling": pl.Int8,
                    "Male": pl.Int8,
                    "Wavy_Hair": pl.Int8,
                    "Black_Hair": pl.Int8,
                    "Wearing_Hat": pl.Int8,
                    "celebrity_id": pl.Int64,
                },
            )
        else:
            self.stems = sorted(
                Path(f).stem for f in os.listdir(img_dir) if f.endswith(".jpg")
            )

    def __len__(self):
        return len(self.stems) if self.test else len(self.data_frame)

    def __getitem__(self, idx):
        if self.test:
            stem = self.stems[idx]
            image = Image.open(Path(self.img_dir) / f"{stem}.jpg").convert("RGB")
            labels = stem  # Just return the stem as label for test set
        else:
            img_name = Path(self.img_dir) / f"{self.data_frame.item(idx, 0)}.jpg"
            image = Image.open(img_name).convert("RGB")
            labels = np.array(self.data_frame.row(idx)[1:], dtype="float32")
        # Transform the image to a tensor and normalize it
        if self.transform:
            image = self.transform(image)
        else:
            image = transforms.ToTensor()(image)
        return image, labels


# Define the transformation pipeline
transform = transforms.Compose(
    [
        transforms.ToTensor(),
    ]
)

# CHALLENGE_DIR = '/kaggle/input/competitions/unipd-deep-learning-2026-challenge-1'
CHALLENGE_DIR = Path(os.getcwd() + "/data/")

TRAIN_CSV_PATH = Path(f"{CHALLENGE_DIR}/train_data.csv")
TRAIN_IMG_DIR  = Path(f"{CHALLENGE_DIR}/train_images")

# Initialize the Dataset and DataLoader
full_train_dataset = CelebDataset(
    img_dir=TRAIN_IMG_DIR,
    csv_file=TRAIN_CSV_PATH,
    transform=transform,
)

# Let's split into training/validation and test sets
train_dataset, val_dataset = random_split(full_train_dataset, [0.8, 0.2], generator=torch.Generator().manual_seed(42))

print(f"Data loaded successfully with {len(train_dataset)} training and {len(val_dataset)} validation samples.")

In [ ]:
# Check the shape of the images
image, labels = full_train_dataset[0]

img_shape = image.shape

print("Image shape:\n",
      f"• Channel \t{img_shape[0]} \n",
      f"• Height: \t{img_shape[1]} \n",
      f"• Width: \t{img_shape[2]}\n")

## MultiTaskCelebNet Class

In [ ]:
NUM_BINARY  = 8
NUM_CLASSES = 501

# Define the Convolutional Neural Network architecture
class MultiTaskCelebNet(nn.Module):
    def __init__(
        self,
        img_shape :tuple = (3, 60, 48),
        conv_filters :list =[],
        kernel_sizes :list =[],
        max_pool_sizes :list =[],
        act_fs :list =[],
        fc_out :int = 2048,
        fc_act :Callable = F.relu,
        dp_rate :float = 0.5,
        verbose=False,
    ):
        super().__init__()

        assert len(conv_filters) == len(kernel_sizes), (
            "length of {conv_filters} and {kernel_sizes} must be same"
        )
        assert len(conv_filters) == len(max_pool_sizes), (
            "length of {conv_filters} and {max_pool_sizes} must be same"
        )
        assert len(conv_filters) == len(act_fs), (
            "length of {conv_filters} and {act_fs} must be same"
        )
        assert callable(fc_act), "fc_act must be a callable (e.g. F.relu, nn.ReLU())"

        self.conv_layers = nn.ModuleList()
        self.max_pools = nn.ModuleList()
        self.in_chan = img_shape[0]     # 3
        self.in_height = img_shape[1]   # 60
        self.in_width = img_shape[2]    # 48
        self.act_fs = act_fs
        self.fc_out = fc_out
        self.fc1_act = fc_act
        self.dp_rate = dp_rate
        self.verbose = verbose

        height_dimension = self.in_height
        width_dimension = self.in_width

        for maxp1, maxp2 in max_pool_sizes:
            height_dimension = height_dimension // maxp1
            width_dimension = width_dimension // maxp2

        self.inp_dim_to_linear = conv_filters[-1] * height_dimension * width_dimension

        for idx in range(len(conv_filters)):
            if idx == 0:
                self.conv_layers.append(
                    Conv2d(
                        self.in_chan,
                        conv_filters[idx],
                        kernel_sizes[idx],
                        padding="same",
                    )
                )
            else:
                self.conv_layers.append(
                    Conv2d(
                        conv_filters[idx - 1],
                        conv_filters[idx],
                        kernel_sizes[idx],
                        padding="same",
                    )
                )
            self.max_pools.append(MaxPool2d(max_pool_sizes[idx]))

        # Fully connected bottleneck + dropout before the two heads
        self.fc1 = Linear(self.inp_dim_to_linear, self.fc_out)
        self.dropout = nn.Dropout(self.dp_rate)

        # Output heads for binary and multi-class classification
        self.binary_head = Linear(self.fc_out, NUM_BINARY)
        self.class_head  = Linear(self.fc_out, NUM_CLASSES)

        pytorch_total_params = sum(
            p.numel() for p in self.parameters() if p.requires_grad
        )
        print(f"The model has {pytorch_total_params} trainable parameters.\n")

    def forward(self, x):
        for idx in range(len(self.conv_layers)):
            from_shape = x.shape[1:]
            x = self.max_pools[idx](self.act_fs[idx](self.conv_layers[idx](x)))
            if self.verbose:
                print(f"From dimension [{from_shape}] to dimension [{x.shape[1:]}]")

        x = torch.flatten(x, start_dim=1)
        x = self.dropout(self.fc1_act(self.fc1(x)))
        return self.binary_head(x), self.class_head(x)


In [ ]:
_m = MultiTaskCelebNet(img_shape=(3, 60, 48), conv_filters=[32], kernel_sizes=[[3, 3]],
                       max_pool_sizes=[[2, 2]], act_fs=[F.relu])
with torch.no_grad():
    _b, _c = _m(torch.zeros(4, 3, 60, 48))
assert isinstance(_b, torch.Tensor) and isinstance(_c, torch.Tensor)
assert _b.shape == (4, 8),   f"binary_logits wrong: {_b.shape}"
assert _c.shape == (4, 501), f"class_logits wrong: {_c.shape}"
assert _m.inp_dim_to_linear == 32 * 30 * 24
del _m, _b, _c
print("All sanity checks passed.")

Choosing batch size and creating DataLoader configuration for training (and validation)

In [ ]:
batch_size = 32

print(f"batch_size: {batch_size}\n")

# DataLoader handles batching, shuffling, and parallel data loading
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(
    f"DataLoader successfully initialized with {len(train_dataset)} training and {len(val_dataset)} validation samples."
)


## Model Creation

Now it's time to inizialize the model using the custom class `MultiTaskCelebNet`

In [ ]:
# Define model hyperparameters and initialize the model
conv_filters = [32, 32, 64, 64, 128, 128]
kernel_sizes = [[3, 3], [3, 3], [3, 3], [3, 3], [3, 3], [3, 3]]
max_pool_sizes = [[1, 1], [2, 2], [1, 1], [2, 2], [1, 1], [2, 2]]
act_fs = [F.relu, F.relu, F.relu, F.relu, F.relu, F.relu]
fc_out = 2048
fc_act = F.relu
dp_rate = 0.5


model = MultiTaskCelebNet(img_shape, conv_filters, kernel_sizes, max_pool_sizes, act_fs, fc_out, fc_act, dp_rate, False).to(
    device
)

# Let's visualize the model
summary(model, input_size=(batch_size, 3, 60, 48))


## Training

In [ ]:
num_epochs = 20
lr = 1e-3
early_stopping_patience = 5

binary_criterion = torch.nn.BCEWithLogitsLoss()
class_criterion  = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# Loss weighting function (optional)
def cls_weight_fn(epoch, bin_loss, cls_loss):
    if epoch == 0:
        return 1 / 8
    else:
        return bin_loss / cls_loss

In [ ]:
def train(
    model,
    dataloader_train,
    dataloader_val,
    optimizer=None,
    binary_criterion=torch.nn.BCEWithLogitsLoss(),
    class_criterion=torch.nn.CrossEntropyLoss(),
    epochs=30,
    hparam_tuning=False,
    cls_weight_fn=None,
    early_stopping_patience=None,
):
    loss_train, loss_val = [], []
    bin_acc_train, bin_acc_val = [], []
    cls_acc_train, cls_acc_val = [], []
    history1 = hl.History()
    canvas1  = hl.Canvas()
    best_val_loss = float('inf')
    no_improve_count = 0
    best_model_state = None

    if optimizer is None:
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    prev_avg_loss_bin = 0.0
    prev_avg_loss_cls = 0.0

    for epoch in range(epochs):
        model.train()
        # Reset metrics and losses for this epoch
        total_bin_acc_train, total_cls_acc_train = 0.0, 0.0
        total_count_train, n_train_batches, total_loss_train = 0, 0, 0
        total_loss_bin, total_loss_cls = 0, 0

        # compute cls weight for this epoch (if a function is provided)
        cls_w = (
            cls_weight_fn(epoch, prev_avg_loss_bin, prev_avg_loss_cls)
            if cls_weight_fn is not None
            else 1.0
        )

        for img, label in dataloader_train:
            img, label = img.to(device), label.to(device)
            optimizer.zero_grad()
            # Calculate logits for binary and multi-class classification heads
            bin_logits, cls_logits = model(img)
            
            # Compute the losses for binary and multi-class classification
            bin_loss = binary_criterion(bin_logits, label[:, :8])
            cls_loss = class_criterion(cls_logits, label[:, 8].long())
            
            # Update total losses for monitoring
            total_loss_bin += bin_loss
            total_loss_cls += cls_loss
            
            # Combine the losses with and weight the multi-class loss
            loss = bin_loss + cls_w * cls_loss

            total_loss_train += loss
            loss.backward()
            optimizer.step()

            total_bin_acc_train += (
                ((torch.sigmoid(bin_logits) >= 0.5) == label[:, :8].bool())
                .float().mean(dim=1).sum().item()
            )
            total_cls_acc_train += (
                (cls_logits.argmax(1) == label[:, 8].long()).sum().item()
            )
            total_count_train += label.size(0)
            n_train_batches += 1

        prev_avg_loss_bin = (total_loss_bin / n_train_batches).item()
        prev_avg_loss_cls = (total_loss_cls / n_train_batches).item()

        avg_loss_train = total_loss_train / n_train_batches
        loss_train.append(avg_loss_train.item())
        bin_acc_train.append(total_bin_acc_train / total_count_train)
        cls_acc_train.append(total_cls_acc_train / total_count_train)

        total_bin_acc_val, total_cls_acc_val = 0.0, 0.0
        total_count_val, n_val_batches, total_loss_val = 0, 0, 0
        with torch.no_grad():
            model.eval()
            for img, label in dataloader_val:
                img, label = img.to(device), label.to(device)
                bin_logits, cls_logits = model(img)
                loss = (binary_criterion(bin_logits, label[:, :8])
                        + cls_w * class_criterion(cls_logits, label[:, 8].long()))
                total_loss_val += loss
                total_bin_acc_val += (
                    ((torch.sigmoid(bin_logits) >= 0.5) == label[:, :8].bool())
                    .float().mean(dim=1).sum().item()
                )
                total_cls_acc_val += (
                    (cls_logits.argmax(1) == label[:, 8].long()).sum().item()
                )
                total_count_val += label.size(0)
                n_val_batches += 1

        avg_loss_val = total_loss_val / n_val_batches
        loss_val.append(avg_loss_val.item())
        bin_acc_val.append(total_bin_acc_val / total_count_val)
        cls_acc_val.append(total_cls_acc_val / total_count_val)

        if not hparam_tuning:
            history1.log(
                epoch,
                train_loss=avg_loss_train,
                train_bin_acc=bin_acc_train[-1],
                train_cls_acc=cls_acc_train[-1],
                val_loss=avg_loss_val,
                val_bin_acc=bin_acc_val[-1],
                val_cls_acc=cls_acc_val[-1],
            )
            with canvas1:
                canvas1.draw_plot([history1["train_loss"], history1["val_loss"]])
                canvas1.draw_plot([history1["train_bin_acc"], history1["val_bin_acc"]])
                canvas1.draw_plot([history1["train_cls_acc"], history1["val_cls_acc"]])
        else:
            print(
                f"epoch: {epoch + 1} -> "
                f"BinAcc: {100 * bin_acc_train[-1]:.2f}%, "
                f"ClsAcc: {100 * cls_acc_train[-1]:.2f}%, "
                f"Loss: {avg_loss_train:.6f}",
                end=" --- ",
            )
            print(
                f"Val_BinAcc: {100 * bin_acc_val[-1]:.2f}%, "
                f"Val_ClsAcc: {100 * cls_acc_val[-1]:.2f}%, "
                f"Val_Loss: {avg_loss_val:.6f}"
            )

        if early_stopping_patience is not None:
            if avg_loss_val.item() < best_val_loss:
                best_val_loss = avg_loss_val.item()
                no_improve_count = 0
                best_model_state = copy.deepcopy(model.state_dict())
            else:
                no_improve_count += 1
                if no_improve_count >= early_stopping_patience:
                    break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return loss_train, bin_acc_train, cls_acc_train, loss_val, bin_acc_val, cls_acc_val


def plot_learning_acc_and_loss(loss_tr, bin_acc_tr, cls_acc_tr, loss_val, bin_acc_val, cls_acc_val):
    plt.figure(figsize=(8, 12))

    plt.subplot(3, 1, 1)
    plt.grid()
    plt.plot(range(len(loss_tr)), loss_tr, label="loss_training")
    plt.plot(range(len(loss_tr)), loss_val, label="loss_validation")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend(loc="best")

    plt.subplot(3, 1, 2)
    plt.grid()
    plt.plot(range(len(bin_acc_tr)), bin_acc_tr, label="bin_acc_training")
    plt.plot(range(len(bin_acc_tr)), bin_acc_val, label="bin_acc_validation")
    plt.xlabel("Epochs")
    plt.ylabel("Binary Attr Accuracy")
    plt.legend(loc="best")

    plt.subplot(3, 1, 3)
    plt.grid()
    plt.plot(range(len(cls_acc_tr)), cls_acc_tr, label="cls_acc_training")
    plt.plot(range(len(cls_acc_tr)), cls_acc_val, label="cls_acc_validation")
    plt.xlabel("Epochs")
    plt.ylabel("Celebrity ID Accuracy")
    plt.legend(loc="best")

    plt.show()


In [ ]:
start = timer()
(loss_train, bin_acc_train, cls_acc_train, loss_val, bin_acc_val, cls_acc_val) = train(
    model,
    train_loader,
    val_loader,
    optimizer,
    epochs=num_epochs,
    binary_criterion=binary_criterion,
    class_criterion=class_criterion,
    cls_weight_fn=cls_weight_fn,
    early_stopping_patience=early_stopping_patience,
)
end = timer()
print(f"Training time in seconds: {(end - start):.1f}")


## Submission

Now it's time to create the submission file using the test images

In [ ]:
TEST_IMG_DIR = Path(f"{CHALLENGE_DIR}/test_images")

test_dataset = CelebDataset(
    img_dir=TEST_IMG_DIR,
    transform=transform,
    test=True,
)

# transform the test images and load them into a DataLoader
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [ ]:
def submit(dataloader_test, model, columns_name,
           output_path=Path("submissions/submission.csv")):
    model.eval()
    all_ids, all_bin, all_cls = [], [], []

    with torch.no_grad():
        for imgs, ids in dataloader_test:
            imgs = imgs.to(device)
            bin_logits, cls_logits = model(imgs)

            bin_preds = (torch.sigmoid(bin_logits) >= 0.5).int().cpu()  # (B, 8)
            cls_preds = cls_logits.argmax(1).cpu()                       # (B,)

            all_ids.extend(ids)
            all_bin.append(bin_preds)
            all_cls.append(cls_preds)

    bin_tensor = torch.cat(all_bin)   # (N, 8)
    cls_tensor = torch.cat(all_cls)   # (N,)

    df = pl.DataFrame(
        {
            columns_name[0]: all_ids,
            **{columns_name[i + 1]: bin_tensor[:, i].numpy() for i in range(8)},
            columns_name[9]: cls_tensor.numpy(),
        },
        schema={
            columns_name[0]: pl.String,
            **{columns_name[i + 1]: pl.Int8 for i in range(8)},
            columns_name[9]: pl.Int64,
        },
    )

    df.write_csv(output_path)
    print(f"Submission saved: {output_path} ({len(df)} rows)")

In [ ]:
columns_name = pl.read_csv(TRAIN_CSV_PATH, n_rows=1).columns  # 10 column names

submit(test_loader, model, columns_name,
       output_path=Path("submissions/submission.csv"))